# Experimento individual e baseline

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="04_treinamento_individual"


1. Selecionar snapshot, algoritmo ridge/dummy e alpha. Uma campanha com um item mantém idempotência e execução independente do kernel.

In [ ]:
from src.data.journey import extract
_,snapshot=extract(cfg);display(snapshot)
model_keys=[params.model_name]
display({"model":params.model_name,"algorithm":params.algorithm,"alpha":params.alpha,"baseline":"DummyRegressor mean"})


2. Submeter e acompanhar treino real opt-in; candidato não é promovido.

In [ ]:
from src.workbench.operations_sdk import campaign
if params.execute_write:
    params.authorize()
    if not params.snapshot_id:
        raise ValueError("Selecione snapshot_id da tabela SQL")
    payload=dict(snapshot_id=str(params.snapshot_id),cycle_code=cfg.cycle,model_keys=model_keys,
        algorithm=params.algorithm,alpha=params.alpha,max_parallel_jobs=params.parallel_jobs,
        requested_by=params.requester,confirm_training=True)
    display({"preflight":payload,"budget":"máximo 2 processos, 2 GiB no container worker; identidades experimentais sintéticas"})
    submitted=campaign(cfg,payload,params.request_key)
    campaign_id=submitted["campaign_id"]
else:
    campaign_id=str(params.campaign_id) if params.campaign_id else ""
if campaign_id:
    state=await_job(cfg,f"/v1/training/campaigns/{campaign_id}",timeout=7200)
    items=call(cfg,"GET",f"/v1/training/campaigns/{campaign_id}/items?limit=500")
    display(state);display(pd.DataFrame(items))
    evidence.update(campaign=state,items=items)
else:
    evidence["training"]="NOT_EXECUTED: escolha campanha existente ou habilite escrita"


3. Comparar métricas, prova de download/checksum e reload feitas pelo job. Se executado, o resultado deve ter três evidências de arquivo por modelo.

In [ ]:
for item in evidence.get("items",[]):
    if item["status"]=="COMPLETED":
        display(item["result"])
        assert len(item["result"]["storage_evidence"])==3
        display(Markdown(f"[Run]({cfg.ui}/#/experiments/{item['result']['experiment_id']}/runs/{item['mlflow_run_id']})"))


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
